# Lab 1.1 – Document-Aware Chunking & Metadata Enrichment

**Course:** Advanced RAG Architecture, Custom Skills & Evaluation on Google Cloud Platform  
**Module:** Week 1 – Precision RAG & Domain Chunking for Compliance Data

This notebook walks through a complete, reproducible pipeline that:

1. Loads the sample SDLC handbook and technical security baseline
2. Performs **document-aware** (heading-preserving) splitting
3. Builds a **Parent-Child** (Small-to-Big) corpus
4. Attaches the full **compliance metadata schema**
5. Persists the result to local JSONL (and optionally BigQuery)
6. Runs the validation checklist required by the lab

The modules used live in `../src/`. You can treat this notebook as the guided tour and the `src/` package as the production-style reference implementation.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make the lab package importable
LAB_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(LAB_ROOT))

from src.chunking import process_document, process_directory, split_markdown_file
from src.metadata import ChunkRecord, validate_records, ChunkType
from src.ingest import write_jsonl, load_jsonl

DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Lab root : {LAB_ROOT}")
print(f"Data dir : {DATA_DIR}")
print(f"Output   : {OUTPUT_DIR}")
print(f"Documents: {list(DATA_DIR.glob('*.md'))}")

## 1. Inspect the source documents

The sample documents are intentionally small but realistic:

- `sdlc_handbook.md` – hierarchical gates and phase requirements
- `security_baseline.md` – numbered controls with explicit Risk Tier / SDLC Phase annotations

In [ ]:
for path in sorted(DATA_DIR.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    print(f"\n{'='*60}")
    print(f"{path.name}  ({len(text)} chars)")
    print("="*60)
    print(text[:600] + ("…" if len(text) > 600 else ""))

## 2. Document-aware splitting (heading boundaries)

`MarkdownHeaderTextSplitter` keeps the heading hierarchy in `Document.metadata`.
This is the foundation of the section path that later becomes the `section` metadata field.

In [ ]:
docs = split_markdown_file(DATA_DIR / "security_baseline.md")

print(f"Produced {len(docs)} heading-based chunks\n")
for i, d in enumerate(docs):
    meta = {k: v for k, v in d.metadata.items() if k.startswith("h")}
    print(f"[{i}] {meta}")
    print(f"     {d.page_content[:120].replace(chr(10), ' ')}…\n")

## 3. Full Parent-Child corpus with metadata enrichment

`process_directory` runs the complete pipeline:

1. Header-aware split for every Markdown file
2. Grouping by hierarchical section path
3. Creation of parent + child records (or standalone when a section has only one unit)
4. Deterministic enrichment of `control_id`, `asset_type`, `risk_tier`, `sdlc_phase`, etc.
5. Schema validation

In [ ]:
corpus = process_directory(DATA_DIR)

print(f"Total chunks : {len(corpus)}")
print(f"  Parents    : {sum(1 for r in corpus if r.chunk_type == 'parent')}")
print(f"  Children   : {sum(1 for r in corpus if r.chunk_type == 'child')}")
print(f"  Standalone : {sum(1 for r in corpus if r.chunk_type == 'standalone')}")
print()

# Quick look at a few records
for r in corpus[:4]:
    print("-" * 50)
    print(f"chunk_id   : {r.chunk_id[:8]}…")
    print(f"doc_type   : {r.doc_type}")
    print(f"section    : {r.section}")
    print(f"control_id : {r.control_id}")
    print(f"asset_type : {r.asset_type}")
    print(f"risk_tier  : {r.risk_tier}")
    print(f"sdlc_phase : {r.sdlc_phase}")
    print(f"chunk_type : {r.chunk_type}")
    print(f"parent_id  : {r.parent_id}")
    print(f"text       : {r.text[:100].replace(chr(10), ' ')}…")

## 4. Validation checklist

The lab requires that every chunk carries a complete metadata record and that Parent-Child links are consistent.

In [ ]:
errors = validate_records(corpus)
if errors:
    print("VALIDATION FAILED")
    for e in errors:
        print("  -", e)
else:
    print("✓ All records pass schema + linkage validation")

# Additional smoke checks
assert all(r.text.strip() for r in corpus), "Empty text found"
assert all(r.doc_type in ("sdlc_handbook", "security_baseline") for r in corpus)

control_ids = sorted({r.control_id for r in corpus if r.control_id})
print(f"✓ Control IDs extracted: {control_ids}")

asset_types = sorted({r.asset_type for r in corpus})
print(f"✓ Asset types present  : {asset_types}")

## 5. Persist the corpus

### 5a. Local JSONL (always available, no GCP required)

In [ ]:
jsonl_path = OUTPUT_DIR / "rag_chunks.jsonl"
write_jsonl(corpus, jsonl_path)
print(f"Wrote {len(corpus)} records → {jsonl_path}")

# Round-trip sanity check
reloaded = load_jsonl(jsonl_path)
assert len(reloaded) == len(corpus)
print("✓ JSONL round-trip OK")

### 5b. BigQuery (optional – requires GCP credentials)

Uncomment and run when you have Application Default Credentials and a project.

In [ ]:
# from src.ingest import write_bigquery
#
# table_ref = write_bigquery(
#     corpus,
#     project_id="YOUR_PROJECT_ID",   # or set GOOGLE_CLOUD_PROJECT
#     dataset_id="rag_lab",
#     table_id="rag_chunks",
#     write_disposition="WRITE_TRUNCATE",  # safe for lab re-runs
# )
# print(f"Loaded into {table_ref}")

## 6. Explore the corpus (examples useful for Lab 1.2)

These simple filters demonstrate why the metadata enrichment is valuable for hybrid retrieval.

In [ ]:
print("=== Critical risk controls ===")
for r in corpus:
    if r.risk_tier == "critical" and r.chunk_type != "parent":
        print(f"  {r.control_id or '(no id)':12} | {r.section[:50]}")

print("\n=== Trust-boundary related chunks ===")
for r in corpus:
    if r.asset_type == "trust_boundary" and r.chunk_type != "parent":
        print(f"  {r.control_id or '(no id)':12} | {r.section[:50]}")

print("\n=== Design-phase material ===")
for r in corpus:
    if r.sdlc_phase == "design" and r.chunk_type != "parent":
        print(f"  {r.control_id or '(no id)':12} | {r.doc_type:18} | {r.section[:40]}")

## 7. Design notes (for your submission)

Record any intentional deviations or extensions here. Examples:

- Why you chose a particular parent-strategy
- Additional metadata fields you added
- How you would handle PDF source documents
- Observations about chunk size vs. retrieval quality

The starter implementation uses a **section-based** Parent-Child strategy and purely deterministic enrichment rules. Both choices prioritise auditability for security workloads.

## Success criteria checklist

- [x] Logical boundaries respected (heading-aware split)
- [x] Required metadata present on every chunk
- [x] Parent-Child linkage correct and resolvable
- [x] Pipeline reproducible (same input → same logical content)
- [x] Corpus usable by Lab 1.2 and the Capstone (JSONL + optional BigQuery)

**Next:** Lab 1.2 will consume `output/rag_chunks.jsonl` (or the BigQuery table) to implement hybrid retrieval + re-ranking.